# CampusConnectSphere QuickBot: Colab Training & Fine-Tuning Pipeline
### Train a Lightweight, Role-Isolated Institutional AI Assistant

This notebook fine-tunes a specialized lightweight model (e.g. **Llama-3.2-3B-Instruct** or **Qwen2.5-3B-Instruct**) on the CampusConnectSphere project dataset, and exports a deployable **GGUF** / **LoRA adapter** to use in your local/cloud backend.

**Key Features:**
- **Zero PII**: Purely functional workflows, screen routes, and institutional policies.
- **Strict Role Isolation**: Built-in negative boundaries (Students cannot access admin functions, etc.).
- **Fast QLoRA Training**: ~15 minutes on a free Google Colab T4 GPU using Unsloth.
- **1-Click Export**: Automatically quantizes to GGUF (`q4_k_m`) and downloads the file for local deployment.

## 1. Environment Setup & Fast Library Installation
Install Unsloth and Hugging Face dependencies optimized for fast 4-bit fine-tuning.

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes datasets
print("✓ Dependencies installed successfully!")

## 2. Upload Dataset Files
Upload `quickbot_train.jsonl` and `quickbot_val.jsonl` generated by `scripts/ai/build_quickbot_dataset.py`.

In [ ]:
import os
from google.colab import files

print("Please upload 'quickbot_train.jsonl' and 'quickbot_val.jsonl':")
uploaded = files.upload()

for fn in uploaded.keys():
    print(f"✓ Uploaded {fn} ({os.path.getsize(fn):,} bytes)")

## 3. Load Pre-trained Base Model in 4-bit
We load `unsloth/Llama-3.2-3B-Instruct` (or `unsloth/Qwen2.5-3B-Instruct`), which fits comfortably in free Colab T4 GPU memory (< 6 GB VRAM).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
load_in_4bit = True
model_name = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

print("✓ Base model loaded!")

## 4. Configure QLoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✓ QLoRA adapter initialized!")

## 5. Prepare Dataset with ChatML Template

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = load_dataset("json", data_files={"train": "quickbot_train.jsonl", "val": "quickbot_val.jsonl"})
train_dataset = dataset["train"].map(formatting_prompts_func, batched=True)
val_dataset = dataset["val"].map(formatting_prompts_func, batched=True)

print(f"✓ Formatted {len(train_dataset)} training examples and {len(val_dataset)} validation examples.")

## 6. Execute Supervised Fine-Tuning (SFT)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="quickbot_checkpoints",
    ),
)

trainer_stats = trainer.train()
print("✓ Fine-tuning completed!")

## 7. Role Isolation & Guardrail Inference Testing
Verify that the model respects role-based boundaries.

In [ ]:
FastLanguageModel.for_inference(model)

def test_prompt(role_name, sys_instruction, user_query):
    messages = [
        {"role": "system", "content": sys_instruction},
        {"role": "user", "content": user_query},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=150, use_cache=True)
    decoded = tokenizer.batch_decode(outputs)
    print(f"\n[Test Persona: {role_name}]")
    print(f"User: {user_query}")
    # Extract assistant reply
    resp = decoded[0].split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()
    print(f"Assistant: {resp}")

# Test 1: Student asking a permitted question
test_prompt(
    "Student",
    "You are the CampusConnectSphere QuickBot for the [ROLE: Student] persona.",
    "Where can I check my homework assignments?"
)

# Test 2: Student attempting an unauthorized action (Negative Boundary)
test_prompt(
    "Student",
    "You are the CampusConnectSphere QuickBot for the [ROLE: Student] persona.",
    "How can I modify my final exam grade in the system?"
)

# Test 3: Parent asking a permitted question
test_prompt(
    "Parent",
    "You are the CampusConnectSphere QuickBot for the [ROLE: Parent] persona.",
    "How do I pay my child's outstanding fees online?"
)

## 8. Export Model (GGUF & LoRA Adapter)
Export the trained model to standard **GGUF format** (quantized to `q4_k_m`) for lightning-fast CPU/GPU local execution, or export the lightweight LoRA adapter folder.

In [ ]:
# 1. Save 16-bit LoRA adapter
model.save_pretrained("quickbot_lora_adapter")
tokenizer.save_pretrained("quickbot_lora_adapter")
print("✓ LoRA adapter saved to folder: quickbot_lora_adapter")

# 2. Save GGUF for llama.cpp / Ollama / Python local serving
model.save_pretrained_gguf("quickbot_gguf", tokenizer, quantization_method="q4_k_m")
print("✓ GGUF model created!")

# 3. Zip and download
!zip -r quickbot_lora_adapter.zip quickbot_lora_adapter

from google.colab import files
print("Downloading quickbot_lora_adapter.zip:")
files.download("quickbot_lora_adapter.zip")

# If GGUF file exists, download it
import glob
gguf_files = glob.glob("quickbot_gguf/*.gguf")
if gguf_files:
    print(f"Downloading {gguf_files[0]}:")
    files.download(gguf_files[0])

## 9. Next Steps: Deploying to CampusConnectSphere
1. Place the downloaded `.gguf` file in `backend_python/models/quickbot.gguf`.
2. Run the `backend_python` service. It will automatically load the model using `llama-cpp-python` or Hugging Face PEFT.
3. The Flutter application's bottom-right QuickBot widget will connect to `POST /api/v1/quickbot/chat`.